# Common Libraries

In [2]:
import os, mujoco, mujoco.viewer
from loop_rate_limiters import RateLimiter
import numpy as np
from mink import (
    Configuration,
    FrameTask,
    PostureTask
)

# Params

In [3]:
myosim_dir_path = "/Users/seojin/myosuite/myosuite/simhive/myo_sim/arm"

# Simulation parameters
duration = 2.0  # seconds
fps = 60
n_frames = int(duration * fps)
dt = 1.0 / fps

solver = "quadprog"
pos_threshold = 1e-4
ori_threshold = 1e-4
max_iters = 20

# Model specification

In [4]:
os.chdir(myosim_dir_path)

model_name = "myoarm.xml"
xml_string = f"""
        <mujoco model="MyoArm with Mocap">
            <include file="{model_name}"/>
            <worldbody>
                <body name="target" pos="0 0 0" quat="0 1 0 0" mocap="true">
                    <geom type="box" size=".15 .15 .15" contype="0" conaffinity="0" rgba=".6 .3 .3 .2"/>
                </body>
            </worldbody>
        </mujoco>
        """

# Initialize model & data

In [5]:
model = mujoco.MjModel.from_xml_string(xml_string)
data = mujoco.MjData(model)

# IK configuration

In [18]:
configuration = Configuration(model)
origin_target = configuration.get_transform_frame_to_world("S_grasp", "site")
origin_target_orientation = origin_target.wxyz_xyz[:4]
origin_target_position = origin_target.wxyz_xyz[4:]

# Task
tasks = [
    end_effector_task := FrameTask(
        frame_name="S_grasp",
        frame_type="site",
        position_cost=1.0,
        orientation_cost=1.0,
        lm_damping=1.0,
    ),
    posture_task := PostureTask(model=model, cost=1e-2),
]

# Renderer
renderer = mujoco.Renderer(configuration.model, height=400, width=600)

In [21]:
# Make target trajectory: move the end-effector in a circular path.
center_point = origin_target_position
r = 0.05
start_angle = 0
end_angle = 2 * np.pi

angles = np.linspace(start_angle, end_angle, n_frames)
target_translations_circular = np.zeros((n_frames, 3))
target_translations_circular[:, 0] = center_point[0] + r * np.cos(angles)
target_translations_circular[:, 1] = center_point[1] + r * np.sin(angles)
target_translations_circular[:, 2] = center_point[2]

target_orientations = np.tile(origin_target_orientation, (n_frames, 1))
target_move_circular = np.concatenate((target_orientations, target_translations_circular), axis=1)

In [26]:
target_move_circular[:, 4:]

array([[-0.10788686,  0.06500652,  0.76948438],
       [-0.10795654,  0.06764529,  0.76948438],
       [-0.10816539,  0.0702767 ,  0.76948438],
       [-0.10851282,  0.07289342,  0.76948438],
       [-0.10899786,  0.07548817,  0.76948438],
       [-0.10961916,  0.07805369,  0.76948438],
       [-0.11037499,  0.08058286,  0.76948438],
       [-0.11126325,  0.0830686 ,  0.76948438],
       [-0.11228146,  0.08550401,  0.76948438],
       [-0.11342678,  0.08788229,  0.76948438],
       [-0.11469601,  0.0901968 ,  0.76948438],
       [-0.11608563,  0.09244111,  0.76948438],
       [-0.11759175,  0.09460895,  0.76948438],
       [-0.11921019,  0.09669429,  0.76948438],
       [-0.12093642,  0.0986913 ,  0.76948438],
       [-0.12276564,  0.10059443,  0.76948438],
       [-0.12469275,  0.10239837,  0.76948438],
       [-0.12671237,  0.1040981 ,  0.76948438],
       [-0.12881889,  0.10568886,  0.76948438],
       [-0.13100642,  0.10716624,  0.76948438],
       [-0.13326888,  0.10852611,  0.769